# CofC Soccer — Match Parsing Notebook
**Your job: drop files, fill in match info, run the cells. That's it.**

---

### What you do each time a new match is ready:
1. **Upload your XML files** to Google Drive (see Step 2)
2. **Fill in the match details** (see Step 3)
3. **Run all cells** — Runtime → Run all
4. **Check the output** and report any ⚠️ warnings

You don't need to understand how the pipeline works.
If something breaks, screenshot the error and send it to Anissa.

---


## Step 1 — Setup
**Run once at the start of every session. Don't change anything here.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
print("✅ Drive mounted")
# You may need to find the shared foleder -> right click -> organize -> add shortcut to drive

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
✅ Drive mounted


In [ ]:
!pip install python-dotenv --quiet
print("✅ Ready")

✅ Ready


In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
# https://drive.google.com/drive/folders/1CX5Tm9R4U5YA8vJCOOJ4FgMfKIqeQV1D?usp=drive_link
PROJECT_ROOT = "/content/drive/MyDrive/CofC_Soccer/data_ingestion_pipeline/"
ROSTER_PATH = os.path.join(PROJECT_ROOT, "roster_2025.csv")
src_path = os.path.join(PROJECT_ROOT, "src")
sys.path.insert(0, src_path)
print(src_path)

from parse_wyscout import parse_sportscode, parse_effective_time, estimate_minutes_played
from parse_spiideo import parse_spiideo
from attribute import calculate_offset, attribute_players
from manifest import load_manifest
print(ROSTER_PATH)
print("✅ Pipeline ready")

/content/drive/MyDrive/CofC_Soccer/data_ingestion_pipeline/src
/content/drive/MyDrive/CofC_Soccer/data_ingestion_pipeline/roster_2025.csv
✅ Pipeline ready


## Step 2 — Drop Your Files

Before running anything, make sure your XML files are in the right place.

**Folder structure:**
```
CofC_Pipeline/
└── matches/
    └── 2025/
        └── 2025-11-02_uncw/              ← folder named: YYYY-MM-DD_opponent
            ├── 2025-11-02_uncw_cfc_sportscode.xml
            ├── 2025-11-02_uncw_cfc_effective_time.xml
            └── 2025-11-02_uncw_spiideo.xml   ← may not exist yet, that's ok
```

**Naming rules:**
- Folder name = `YYYY-MM-DD_opponentshortname` (lowercase, no spaces)
- Files must start with the same folder name
- Spiideo file is optional — pipeline still works without it

Once files are in place, run the cell below to confirm everything is found.


In [ ]:
# ── EDIT THIS: set the match slug ──────────────────────────────────────
MATCH_SLUG = "2025-08-22_south_carolina"    # ← change this to your match folder name
SEASON     = "2025"
# ───────────────────────────────────────────────────────────────────────

MATCHES_DIR = Path(PROJECT_ROOT) / "matches"
match_dir   = MATCHES_DIR / SEASON / MATCH_SLUG

sportscode_file    = match_dir / f"{MATCH_SLUG}_cfc_sportscode.xml"
effective_time_file= match_dir / f"{MATCH_SLUG}_cfc_effective_time.xml"
spiideo_file       = match_dir / f"{MATCH_SLUG}_spiideo.xml"

print(f"Checking files for: {MATCH_SLUG}")
print()

checks = [
    (sportscode_file,     "Wyscout Sportscode XML",   True),
    (effective_time_file, "Wyscout Effective Time XML",True),
    (spiideo_file,        "Spiideo COUG tags XML",     False),  # optional
]

all_required_ok = True
for path, label, required in checks:
    if path.exists():
        print(f"  ✅  {label}")
    elif required:
        print(f"  ❌  {label}  ← MISSING — check file name and location")
        all_required_ok = False
    else:
        print(f"  ⚠️   {label}  ← not found (pipeline will run without COUG scores)")

print()
if all_required_ok:
    print("✅ Files look good — continue to Step 3")
else:
    print("❌ Fix missing files before continuing.")
    print("   Check: is the folder named correctly? Are the files named correctly?")

Checking files for: 2025-08-22_south_carolina

  ✅  Wyscout Sportscode XML
  ✅  Wyscout Effective Time XML
  ⚠️   Spiideo COUG tags XML  ← not found (pipeline will run without COUG scores)

✅ Files look good — continue to Step 3


In [ ]:
from pathlib import Path

# Reconstruct the problematic file path
file_to_check = Path(PROJECT_ROOT) / "matches" / SEASON / MATCH_SLUG / f"{MATCH_SLUG}_cfc_sportscode.xml"

print(f"Checking for file: {file_to_check}")
if file_to_check.exists():
    print("✅ File exists in Google Drive.")
else:
    print("❌ File does NOT exist at this path in Google Drive.")
    print("   Please verify the file name and its location in your Shared Drive.")

Checking for file: /content/drive/MyDrive/CofC_Soccer/data_ingestion_pipeline/matches/2025/2025-08-22_south_carolina/2025-08-22_south_carolina_cfc_sportscode.xml
✅ File exists in Google Drive.


## Step 3 — Fill In Match Details

Fill in the details for this match. This gets added to the manifest.


In [ ]:
# ── EDIT ALL OF THESE ──────────────────────────────────────────────────
OPPONENT      = "South Carolina"           # full opponent name
COMPETITION   = "Non-Conference"            # CAA | Non-Conference | Preseason
VENUE         = "Patriots Point" # where the match was played
COFC_GOALS    = 2              # CofC goals scored
OPP_GOALS     = 3               # opponent goals scored
MANUAL_OFFSET = None             # leave as None unless told otherwise
# ───────────────────────────────────────────────────────────────────────

result = "W" if COFC_GOALS > OPP_GOALS else ("L" if COFC_GOALS < OPP_GOALS else "D")

print(f"Match:       CofC vs {OPPONENT}")
print(f"Date:        {MATCH_SLUG.split('_')[0]}")
print(f"Competition: {COMPETITION}")
print(f"Venue:       {VENUE}")
print(f"Result:      {result}  ({COFC_GOALS}–{OPP_GOALS})")
print()
print("✅ Details set — continue to Step 4")

Match:       CofC vs South Carolina
Date:        2025-08-22
Competition: Non-Conference
Venue:       Patriots Point
Result:      L  (2–3)

✅ Details set — continue to Step 4


## Step 4 — Run the Pipeline

This cell does all the work. Run it and wait for the ✅ at the bottom.

**What it does:**
- Parses the Wyscout XML → player events
- Parses the Spiideo XML (if available) → COUG moments
- Links COUG moments to players using timestamps
- Saves CSVs to your Drive

**If you see ⚠️ Attribution rate is low:** note the number and message Anissa.


In [ ]:
output_dir = Path(PROJECT_ROOT) / "outputs" / SEASON / MATCH_SLUG
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Processing: {MATCH_SLUG}")
print("=" * 55)

# 1. Parse Wyscout
print("\n[1/4] Parsing Wyscout XML...")
wyscout_data = parse_sportscode(sportscode_file, roster_path=ROSTER_PATH)
df_players     = pd.DataFrame(wyscout_data["player_events"])
df_team_events = pd.DataFrame(wyscout_data["team_events"])
print(f"      {len(df_players)} player events | {df_players['name'].nunique()} players")

# 2. Parse Spiideo (optional)
has_spiideo = spiideo_file.exists()
if has_spiideo:
    print("\n[2/4] Parsing Spiideo XML...")
    spiideo_data = parse_spiideo(spiideo_file)
    df_spiideo   = pd.DataFrame(spiideo_data["coug_events"])
    print(f"      {len(df_spiideo)} COUG events found")
else:
    print("\n[2/4] No Spiideo file — skipping COUG attribution")
    spiideo_data = None
    df_spiideo   = pd.DataFrame()

# 3. Attribution
if has_spiideo and spiideo_data:
    print("\n[3/4] Linking COUG moments to players...")
    offset = calculate_offset(
        wyscout_halves           = wyscout_data["halves"],
        spiideo_all_events       = spiideo_data["all_events"],
        spiideo_recording_start  = None,
        manual_offset            = MANUAL_OFFSET,
    )
    first_half_start = wyscout_data["halves"].get("first_start", 2.0)
    attributed = attribute_players(
        coug_events    = spiideo_data["coug_events"],
        player_events  = wyscout_data["player_events"],
        offset         = offset,
        first_half_start = first_half_start,
    )
    df_attributed = pd.DataFrame([{
        "match":          MATCH_SLUG,
        "minute":         round(ev.get("match_minute", 0), 1),
        "category":       ev["category"],
        "subtype":        ev["subtype"],
        "player_name":    ev["player"]["name"] if ev.get("player") else None,
        "outcome":        ev["player"]["outcome"] if ev.get("player") else None,
        "attribution_score": ev.get("attribution_score", 0),
    } for ev in attributed])

    attr_rate = df_attributed["player_name"].notna().mean() * 100
    print(f"      Attribution rate: {attr_rate:.0f}%")
    if attr_rate < 50:
        print(f"      ⚠️  Low attribution — consider setting MANUAL_OFFSET")
        print(f"         Message Anissa with this number and the offset value: {offset:.1f}s")
else:
    df_attributed = pd.DataFrame()
    attr_rate = 0

# 4. Save CSVs
print("\n[4/4] Saving CSVs...")

saved = []

players_path = output_dir / f"{MATCH_SLUG}_players.csv"
df_players.to_csv(players_path, index=False)
saved.append(f"      ✅ {players_path.name}")

if not df_spiideo.empty:
    spiideo_path = output_dir / f"{MATCH_SLUG}_spiideo_events.csv"
    df_spiideo.to_csv(spiideo_path, index=False)
    saved.append(f"      ✅ {spiideo_path.name}")

if not df_attributed.empty:
    attr_path = output_dir / f"{MATCH_SLUG}_attributed.csv"
    df_attributed.to_csv(attr_path, index=False)
    saved.append(f"      ✅ {attr_path.name}")

for s in saved:
    print(s)

# Summary
print()
print("=" * 55)
print(f"DONE — {MATCH_SLUG}")
print(f"  Players parsed:    {df_players['name'].nunique()}")
if has_spiideo:
    print(f"  COUG events:       {len(df_spiideo)}")
    print(f"  Attribution rate:  {attr_rate:.0f}%")
    if attr_rate < 50:
        print(f"  ⚠️  Message Anissa — attribution is low")
else:
    print(f"  COUG events:       (no Spiideo file)")
print(f"  Files saved to:    outputs/{SEASON}/{MATCH_SLUG}/")
print("=" * 55)

Processing: 2025-08-22_south_carolina

[1/4] Parsing Wyscout XML...
  Roster filter: 28 CofC players loaded
  Sportscode: 741 player events, 914 team events | halves: {'first_start': 2.0, 'second_start': 2876.0, 'first_end': 2877.0, 'second_end': 6465.0} | 868 opponent events filtered out
      741 player events | 16 players

[2/4] No Spiideo file — skipping COUG attribution

[4/4] Saving CSVs...
      ✅ 2025-08-22_south_carolina_players.csv

DONE — 2025-08-22_south_carolina
  Players parsed:    16
  COUG events:       (no Spiideo file)
  Files saved to:    outputs/2025/2025-08-22_south_carolina/


## Step 5 — Check the Output

Run this cell to preview what was saved.
If the tables look wrong or empty, message Anissa with a screenshot.


In [ ]:
print(f"Output for: {MATCH_SLUG}")
print()

# Players
if players_path.exists():
    df_check = pd.read_csv(players_path)
    print(f"Players ({len(df_check)} events, {df_check['name'].nunique()} players):")
    print(df_check[["name", "start", "end", "outcome"]].head(8).to_string(index=False))
    print()

# COUG attribution
attr_path = output_dir / f"{MATCH_SLUG}_attributed.csv"
if attr_path.exists():
    df_check = pd.read_csv(attr_path)
    print(f"COUG events attributed ({len(df_check)} total):")
    print(df_check[["minute", "category", "subtype", "player_name"]].head(10).to_string(index=False))
    print()
    unattr = df_check["player_name"].isna().sum()
    if unattr > 0:
        print(f"⚠️  {unattr} events could not be attributed to a player")
        print(f"   This is normal for some events. Message Anissa if it's more than half.")
    else:
        print("✅ All COUG events attributed to players")
else:
    print("No attribution file — either no Spiideo file was provided, or Step 4 failed.")
    print("Check Step 4 output for errors.")

Output for: 2025-08-22_south_carolina

Players (741 events, 16 players):
   name  start   end outcome
T. Nero   46.0  57.0 Unknown
T. Nero   48.0  59.0 Unknown
T. Nero  135.0 146.0 Unknown
T. Nero  175.0 186.0 Unknown
T. Nero  248.0 259.0 Unknown
T. Nero  373.0 384.0 Unknown
T. Nero  406.0 417.0 Unknown
T. Nero  519.0 530.0 Unknown

No attribution file — either no Spiideo file was provided, or Step 4 failed.
Check Step 4 output for errors.


# Step 6 — Compute COUGs Table and save coug_scores CSV

*   Always runs — produces a coug_scores CSV for every match regardless of whether Spiideo attribution is available.

*   Wyscout-only:  all players seeded with minutes played, zero COUG points
*   With Spiideo:  ASET/PEAK points added on top of Wyscout base

*   ASET, PEAK, and total COUG scores using the trial_1 weights.
*   Saves {MATCH_SLUG}_coug_scores.csv to the same output folder.

**NOTE:** Derived metrics (clean sheet, concede goal, goal on field, positional) are flagged as pending — they require substitution logic built in the DB ingest layer.


In [ ]:
# ── Step 6 — Compute COUGs Table and save coug_scores CSV ──────────────────
#
# Two-pass scoring:
#   Pass 1 — Wyscout labels from _players.csv (always runs)
#   Pass 2 — Spiideo attribution from _attributed.csv (if available)
#
# Roster-aware filtering:
#   - GKs skip Goal / Free kick goal labels (those are opponent goals)
#   - Positional metrics only credit the correct position group
#
# Final score = Wyscout base + Spiideo supplement
# spiideo_included column flags which matches have full coverage.

import pandas as pd
import ast
import csv
from pathlib import Path
from collections import defaultdict

# ── Weights (trial_1) ──────────────────────────────────────────────────────
WEIGHTS_VERSION   = "trial_1"
MIN_MINUTES_PER90 = 45

# ── Wyscout label → (category, weight) ────────────────────────────────────
WYSCOUT_LABEL_WEIGHTS = {
    # ASET — Defense
    "Vol_Interception":  ("ASET", 0.25),
    "Tackles":           ("ASET", 0.25),   # Plus only
    "Clearances":        ("ASET", 0.20),   # Plus only
    "Anticipated":       ("ASET", 0.25),   # Plus only
    "Anticipation":      ("ASET", 0.25),   # Plus only
    "Pressing duel":     ("ASET", 0.10),   # Non-minus
    "Loose ball duel":   ("ASET", 0.15),   # Non-minus
    "Defensive duel":    ("ASET", 0.10),   # Non-minus
    "1VS1":              ("ASET", 0.10),   # Non-minus

    # PEAK — Offense
    "Goal":              ("PEAK", 3.00),   # Always — GK skipped via roster
    "Assists":           ("PEAK", 2.00),   # Always
    "Key passes":        ("PEAK", 0.20),   # Plus only
    "Smart pass":        ("PEAK", 0.20),   # Plus only
    "Smart passes":      ("PEAK", 0.20),   # Plus only
    "Opportunity":       ("PEAK", 0.20),   # Plus only

    # Set Piece
    "Saves":             ("ASET", 1.00),   # Always (GK only via roster)
    "Free kick goal":    ("PEAK", 1.00),   # Always — GK skipped via roster
    "Free kick shot":    ("PEAK", 0.20),   # Plus only

    # Positional — applied only to correct position group (see below)
    "Aerial duels":      ("ASET", 0.50),   # CB only
    "Cross":             ("PEAK", 0.50),   # WB only — 8 crosses threshold handled below
    "Shots":             ("PEAK", 0.50),   # FWD only (W, F, F/W)
}

# Outcome filter tiers
PLUS_ONLY = {
    "Clearances", "Tackles", "Anticipated", "Anticipation",
    "Key passes", "Smart pass", "Smart passes",
    "Opportunity", "Free kick shot",
}
NON_MINUS = {
    "Defensive duel", "1VS1", "Pressing duel", "Loose ball duel",
}
ALWAYS_COUNT = {
    "Goal", "Assists", "Vol_Interception", "Saves", "Free kick goal",
}

# Positional restrictions
CB_ONLY    = {"Aerial duels"}
WB_ONLY    = {"Cross"}
FWD_ONLY   = {"Shots"}
GK_SKIP    = {"Goal", "Free kick goal"}
GK_ONLY    = {"Saves"}

CB_POS     = {"CB"}
WB_POS     = {"WB", "RB", "LB"}
FWD_POS    = {"F", "W", "F/W", "WF"}
GK_POS     = {"GK"}

# ── Spiideo subtype → (category, weight) ──────────────────────────────────
SUBTYPE_WEIGHTS = {
    "Clearance from Danger":        ("ASET", 0.50),
    "Block in Box":                 ("ASET", 0.20),
    "Successful Counter Press":     ("ASET", 0.20),
    "Possession Regain":            ("ASET", 0.25),
    "Assist":                       ("PEAK", 2.00),
    "Goal (scorer)":                ("PEAK", 3.00),
    "Punish Action after Regain":   ("PEAK", 0.20),
    "Set Piece Goal (1st phase)":   ("PEAK", 1.00),
    "Set Piece Goal (2nd phase)":   ("PEAK", 0.50),
    "Win 1st Header (defensive)":   ("ASET", 0.25),
    "Win 1st Header (offensive)":   ("PEAK", 0.25),
    "Freekick Save/Block":          ("ASET", 1.00),
    "Penalty Save":                 ("ASET", 3.00),
}

# ── Load roster for position-aware filtering ───────────────────────────────
roster_pos = {}   # {player_name: pos}
roster_grp = {}   # {player_name: group}

if Path(ROSTER_PATH).exists():
    with open(ROSTER_PATH, newline="") as f:
        for row in csv.DictReader(f):
            if row.get("name"):
                name = row["name"].strip()
                roster_pos[name] = row.get("pos",   "").strip()
                roster_grp[name] = row.get("group", "").strip()
    print(f"  Roster loaded: {len(roster_pos)} players")
else:
    print(f"  ⚠️  Roster not found at {ROSTER_PATH} — positional filters disabled")

# ── Load players — always required ────────────────────────────────────────
players_path = output_dir / f"{MATCH_SLUG}_players.csv"

if not players_path.exists():
    print("⚠️  No players file — skipping COUGs Table. Run Step 4 first.")
else:
    df_players = pd.read_csv(players_path)

    # ── Minutes played ─────────────────────────────────────────────────────
    minutes_lookup = {}
    for name, grp in df_players.groupby("name"):
        span = (grp["start"].max() - grp["start"].min()) / 60
        minutes_lookup[name] = round(max(5.0, min(span, 100.0)), 1)

    # ── Cross counts per player (for WB threshold logic) ──────────────────
    cross_counts = defaultdict(int)
    for _, row in df_players.iterrows():
        try:
            labels = ast.literal_eval(str(row["labels"]))
            if "Cross" in labels:
                cross_counts[row["name"]] += 1
        except Exception:
            pass

    # ── Seed scores at zero for all players ───────────────────────────────
    scores = {
        name: {
            "aset": 0.0, "peak": 0.0, "event_count": 0,
            "aset_breakdown": defaultdict(float),
            "peak_breakdown": defaultdict(float),
        }
        for name in minutes_lookup
    }

    # ══════════════════════════════════════════════════════════════════════
    # PASS 1 — Wyscout labels
    # ══════════════════════════════════════════════════════════════════════
    wyscout_scored = 0

    for _, row in df_players.iterrows():
        player  = row["name"]
        outcome = str(row.get("outcome", "Unknown"))
        pos     = roster_pos.get(player, "")

        try:
            labels = ast.literal_eval(str(row["labels"]))
        except Exception:
            continue

        for label in labels:
            if label not in WYSCOUT_LABEL_WEIGHTS:
                continue

            # ── Positional restrictions ────────────────────────────────
            if label in GK_SKIP  and pos in GK_POS:  continue
            if label in GK_ONLY  and pos not in GK_POS: continue
            if label in CB_ONLY  and pos not in CB_POS: continue
            if label in FWD_ONLY and pos not in FWD_POS: continue
            if label in WB_ONLY:
                # WB only AND must hit 8-cross threshold for the match
                if pos not in WB_POS: continue
                if cross_counts.get(player, 0) < 8: continue

            # ── Outcome filtering ──────────────────────────────────────
            if label in PLUS_ONLY   and outcome != "Plus":  continue
            if label in NON_MINUS   and outcome == "Minus": continue
            # ALWAYS_COUNT: no filtering

            category, weight = WYSCOUT_LABEL_WEIGHTS[label]
            acc = scores[player]
            acc["event_count"] += 1
            wyscout_scored += 1

            if category == "ASET":
                acc["aset"] += weight
                acc["aset_breakdown"][f"WY:{label}"] += weight
            else:
                acc["peak"] += weight
                acc["peak_breakdown"][f"WY:{label}"] += weight

    print(f"  Wyscout pass: {wyscout_scored} events scored")

    # ══════════════════════════════════════════════════════════════════════
    # PASS 2 — Spiideo attribution (if available)
    # ══════════════════════════════════════════════════════════════════════
    attr_path      = output_dir / f"{MATCH_SLUG}_attributed.csv"
    spiideo_scored = 0
    unmatched      = set()
    has_spiideo    = False

    if attr_path.exists():
        df_attr = pd.read_csv(attr_path)
        df_attr = df_attr[df_attr["player_name"].notna()].copy()

        if not df_attr.empty:
            has_spiideo = True
            for _, row in df_attr.iterrows():
                subtype = str(row["subtype"]).strip()
                player  = row["player_name"]

                if subtype not in SUBTYPE_WEIGHTS:
                    unmatched.add(subtype)
                    continue

                category, weight = SUBTYPE_WEIGHTS[subtype]

                if player not in scores:
                    scores[player] = {
                        "aset": 0.0, "peak": 0.0, "event_count": 0,
                        "aset_breakdown": defaultdict(float),
                        "peak_breakdown": defaultdict(float),
                    }
                    minutes_lookup[player] = 90.0

                acc = scores[player]
                acc["event_count"] += 1
                spiideo_scored += 1

                if category == "ASET":
                    acc["aset"] += weight
                    acc["aset_breakdown"][f"SP:{subtype}"] += weight
                else:
                    acc["peak"] += weight
                    acc["peak_breakdown"][f"SP:{subtype}"] += weight

        if unmatched:
            print(f"  ⚠️  Unmatched Spiideo subtypes: {unmatched}")

    print(f"  Spiideo pass: {spiideo_scored} events scored"
          if has_spiideo else "  Spiideo pass: no file — skipped")

    # ── Build output rows ──────────────────────────────────────────────────
    def per90(pts, mins):
        return round((pts / mins) * 90, 3) if mins >= MIN_MINUTES_PER90 else None

    rows = []
    for player, acc in sorted(
        scores.items(),
        key=lambda x: -(x[1]["aset"] + x[1]["peak"])
    ):
        aset    = round(acc["aset"], 2)
        peak    = round(acc["peak"], 2)
        total   = round(aset + peak, 2)
        minutes = minutes_lookup.get(player, 90.0)

        rows.append({
            "match":            MATCH_SLUG,
            "player":           player,
            "minutes":          minutes,
            "aset":             aset,
            "peak":             peak,
            "total":            total,
            "aset_per90":       per90(aset, minutes),
            "peak_per90":       per90(peak, minutes),
            "total_per90":      per90(total, minutes),
            "event_count":      acc["event_count"],
            "aset_breakdown":   dict(acc["aset_breakdown"]),
            "peak_breakdown":   dict(acc["peak_breakdown"]),
            "weights_version":  WEIGHTS_VERSION,
            "spiideo_included": has_spiideo,
        })

    df_scores = pd.DataFrame(rows)

    # ── Save ───────────────────────────────────────────────────────────────
    scores_path = output_dir / f"{MATCH_SLUG}_coug_scores.csv"
    df_scores.to_csv(scores_path, index=False)

    # ── Preview ────────────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"COUGs TABLE — {MATCH_SLUG}")
    print(f"{'='*55}")
    print(df_scores[["player","minutes","aset","peak","total"]].to_string(index=False))
    print(f"\n✅ Saved → {scores_path.name}")
    print(f"   {len(df_scores)} players | weights: {WEIGHTS_VERSION}")
    print(f"   Wyscout events: {wyscout_scored} | "
          f"Spiideo events: {spiideo_scored} | "
          f"Full coverage: {has_spiideo}")
    if not has_spiideo:
        print(f"   Pending: Clearance from Danger, Counter Press, Headers")

  Roster loaded: 28 players
  Wyscout pass: 252 events scored
  Spiideo pass: no file — skipped

COUGs TABLE — 2025-08-22_south_carolina
     player  minutes  aset  peak  total
   C. Duske    100.0 12.25   0.0  12.25
 B. Bagshaw    100.0  2.25   7.0   9.25
   E. White    100.0  3.50   2.0   5.50
    N. Gold    100.0  4.00   1.0   5.00
  P. Dashin    100.0  1.25   3.0   4.25
  H. Walker     37.1  0.45   3.0   3.45
S. Bendvold     64.0  1.20   2.0   3.20
    T. Nero    100.0  2.65   0.0   2.65
  M. Lenert     70.7  1.60   1.0   2.60
  R. Watson     81.5  2.50   0.0   2.50
 J. Barrett    100.0  2.00   0.0   2.00
  I. Salifu    100.0  1.90   0.0   1.90
    L. Gill     51.6  1.60   0.0   1.60
  C. Hughes     33.4  1.40   0.0   1.40
   A. Duran     11.0  0.50   0.0   0.50
     R. Ray     17.5  0.40   0.0   0.40

✅ Saved → 2025-08-22_south_carolina_coug_scores.csv
   16 players | weights: trial_1
   Wyscout events: 252 | Spiideo events: 0 | Full coverage: False
   Pending: Clearance from Dang

## Step 7 — You're Done ✅


**Send Anissa:**
- A quick message that the match is processed
- The attribution rate (from Step 4)
- Any ⚠️ warnings with a screenshot

**If you have more matches to process:**
- Go back to Step 2
- Change `MATCH_SLUG` to the next match folder name
- Change the match details in Step 3
- Re-run Steps 4 and 5

---

### Common problems

| What you see | What to do |
|---|---|
| `❌ MISSING` in Step 2 | Check file name — it must match the folder name exactly |
| Attribution rate < 50% | Message Anissa with the offset number from Step 4 |
| COUGs Table looks wrong | Check Step 6 warnings — unmatched subtypes or roster not found |
| All scores are zero | No Spiideo file yet — Wyscout-only scores pending |
| `ModuleNotFoundError` | Re-run Step 1 from the top |
| Any other error | Screenshot and message Anissa |

---
*CofC Soccer Analytics — Operator Notebook v2*
